# 🛒 Order Cancellation Risk Prediction in E-Commerce

## Course: MIS444 — Predictive Analytics in Business

**Problem Type:** Classification — Predicting whether an e-commerce order is at high risk of cancellation.

**Dataset:** `E-commerce data.csv`

**Business Objective:**  
Use transaction-level e-commerce data to predict **order cancellation risk** so the business can identify risky orders early, improve customer support prioritization, reduce operational loss, and improve fulfillment planning.

**Target Variable:**  
`IsCancelled` — derived from `InvoiceNo`. In this dataset, invoices beginning with **"C"** represent cancelled transactions.

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Data Collection

The dataset contains online retail transaction records with invoice, product, quantity, price, customer, date, and country information.

Original columns:
- `InvoiceNo`
- `StockCode`
- `Description`
- `Quantity`
- `InvoiceDate`
- `UnitPrice`
- `CustomerID`
- `Country`

Since cancellation is recorded at invoice level, this project converts the raw transaction-line dataset into an **invoice/order-level dataset** before modeling.

## Feature Description Table

| Feature Name | Type | Description |
| :----------- | :--- | :---------- |
| `TotalQuantity` | Numerical | Sum of absolute quantities for all items in an order. |
| `TotalAmount` | Numerical | Total monetary value of the order. |
| `AvgUnitPrice` | Numerical | Average unit price of items in the order. |
| `MaxUnitPrice` | Numerical | Maximum unit price of an item in the order. |
| `UniqueProducts` | Numerical | Number of distinct products in the order. |
| `TotalLines` | Numerical | Total number of line items (transactions) within an order. |
| `CustomerKnown` | Numerical | Binary indicator: 1 if `CustomerID` is present, 0 otherwise. |
| `WeekendOrder` | Numerical | Binary indicator: 1 if the order was placed on a weekend, 0 otherwise. |
| `CustomerOrderCount` | Numerical | Total number of orders placed by the customer. |
| `CustomerCancellationRate` | Numerical | Proportion of past orders cancelled by the customer. |
| `CustomerAvgAmount` | Numerical | Average total amount of past orders placed by the customer. |
| `InvoiceHour` | Numerical | Hour of the day when the invoice was issued (0-23). |
| `InvoiceDayOfWeek` | Categorical | Day of the week when the invoice was issued (e.g., 'Monday', 'Tuesday'). |
| `InvoiceMonth` | Numerical | Month when the invoice was issued (1-12). |
| `InvoiceQuarter` | Numerical | Quarter of the year when the invoice was issued (1-4). |
| `Country` | Categorical | Country where the order was placed. |

In [ ]:
# ── Importing necessary libraries ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)

from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance

sns.set(style='whitegrid')
pd.set_option('display.max_columns', 100)

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────
# Colab / Google Drive path used first. Local fallback is included for portability.

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    file_path = '/content/drive/MyDrive/MIS444 26 Amit/Dataset/E-commerce data.csv'
except Exception:
    file_path = 'E-commerce data.csv'

# The file contains special currency/text characters, so latin1 is safer than utf-8.
ecom = pd.read_csv(file_path, encoding='latin1')

print("Dataset loaded successfully!")
print(f"Shape: {ecom.shape}")
display(ecom.head())

---
## 2. Data Preprocessing

Preprocessing ensures the data is clean and model-ready.

Steps:
- Inspect missing values
- Create cancellation target
- Convert dates to datetime
- Remove duplicate rows
- Handle invalid values
- Build invoice/order-level features
- Avoid direct leakage from cancellation indicators

In [ ]:
# ── Step 2.1 : Inspect missing values ──────────────────────────────────────
print("=== Missing Values Per Column ===")
print(ecom.isnull().sum())
print(f"\nTotal missing values: {ecom.isnull().sum().sum()}")

print("\n=== Dataset Info ===")
ecom.info()

In [ ]:
# ── Step 2.2 : Visualise missing values ────────────────────────────────────
missing = ecom.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=missing.index, y=missing.values,palette='Reds_r' )
plt.title('Missing Values by Column')
plt.xlabel('Column')
plt.ylabel('Missing Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# ── Step 2.3 : Create cancellation target ──────────────────────────────────
# In this dataset, cancelled invoices begin with 'C'.
ecom['InvoiceNo_Str'] = ecom['InvoiceNo'].astype(str)
ecom['IsCancelled'] = ecom['InvoiceNo_Str'].str.startswith('C').astype(int)

print("Cancellation distribution in raw transaction lines:")
print(ecom['IsCancelled'].value_counts())
print("\nCancellation percentage:")
print((ecom['IsCancelled'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# ── Step 2.4 : Convert InvoiceDate to datetime ─────────────────────────────
ecom['InvoiceDate'] = pd.to_datetime(ecom['InvoiceDate'], errors='coerce')

print("Invalid InvoiceDate values:", ecom['InvoiceDate'].isnull().sum())
print("Date range:", ecom['InvoiceDate'].min(), "to", ecom['InvoiceDate'].max())

In [ ]:
# ── Step 2.5 : Check and remove duplicate rows ─────────────────────────────
duplicates = ecom.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")

if duplicates > 0:
    ecom.drop_duplicates(inplace=True)
    print(f"Shape after duplicate removal: {ecom.shape}")
else:
    print("No duplicate rows found.")

In [ ]:
# ── Step 2.6 : Check invalid / unusual numeric values ──────────────────────
print("=== Numeric Value Check ===")
print("Negative Quantity rows:", (ecom['Quantity'] < 0).sum())
print("Negative UnitPrice rows:", (ecom['UnitPrice'] < 0).sum())
print("Zero UnitPrice rows:", (ecom['UnitPrice'] == 0).sum())

# For cancellation risk modeling, negative Quantity is a cancellation signal.
# To avoid leakage, we use absolute quantity when creating order-level features.
ecom['AbsQuantity'] = ecom['Quantity'].abs()
ecom['AbsLineAmount'] = ecom['AbsQuantity'] * ecom['UnitPrice']
ecom['CustomerKnown'] = ecom['CustomerID'].notna().astype(int)

In [ ]:
# ── Step 2.7 : Build invoice/order-level dataset ───────────────────────────
# Cancellation is invoice-level, so raw line items are aggregated into one row per order.

def mode_or_first(series):
    mode_value = series.mode(dropna=True)
    if len(mode_value) > 0:
        return mode_value.iloc[0]
    return series.iloc[0]

orders = ecom.groupby('InvoiceNo_Str').agg(
    IsCancelled=('IsCancelled', 'max'),
    CustomerID=('CustomerID', 'first'),
    Country=('Country', mode_or_first),
    InvoiceDate=('InvoiceDate', 'min'),
    TotalQuantity=('AbsQuantity', 'sum'),
    TotalAmount=('AbsLineAmount', 'sum'),
    AvgUnitPrice=('UnitPrice', 'mean'),
    MaxUnitPrice=('UnitPrice', 'max'),
    UniqueProducts=('StockCode', 'nunique'),
    TotalLines=('StockCode', 'count'),
    CustomerKnown=('CustomerKnown', 'max')
).reset_index()

print("Order-level dataset created successfully!")
print(f"Shape: {orders.shape}")
display(orders.head())

In [ ]:
# ── Step 2.8 : Feature engineering from date and customer information ──────
orders['InvoiceHour'] = orders['InvoiceDate'].dt.hour
orders['InvoiceDayOfWeek'] = orders['InvoiceDate'].dt.day_name()
orders['InvoiceMonth'] = orders['InvoiceDate'].dt.month
orders['InvoiceQuarter'] = orders['InvoiceDate'].dt.quarter
orders['WeekendOrder'] = orders['InvoiceDate'].dt.dayofweek.isin([5, 6]).astype(int)

# CustomerID itself is an identifier, not a true business feature.
# Instead, use safer customer-level order history features calculated from the available dataset.
customer_summary = orders.groupby('CustomerID').agg(
    CustomerOrderCount=('InvoiceNo_Str', 'count'),
    CustomerCancellationRate=('IsCancelled', 'mean'),
    CustomerAvgAmount=('TotalAmount', 'mean')
).reset_index()

orders = orders.merge(customer_summary, on='CustomerID', how='left')

# Missing CustomerID means customer-level history is unknown.
orders['CustomerOrderCount'] = orders['CustomerOrderCount'].fillna(0)
orders['CustomerCancellationRate'] = orders['CustomerCancellationRate'].fillna(0)
orders['CustomerAvgAmount'] = orders['CustomerAvgAmount'].fillna(orders['TotalAmount'].median())

print("Feature engineering completed!")
display(orders.head())

In [ ]:
# ── Step 2.9 : Drop direct identifiers and leakage-prone columns ───────────
# We do NOT use InvoiceNo_Str because 'C' directly reveals cancellation.
# We do NOT use raw negative Quantity because it directly signals cancellation.
# InvoiceDate is converted into useful date features, then removed.

model_data = orders.drop(columns=['InvoiceNo_Str', 'InvoiceDate', 'CustomerID'])

print("Final modeling dataset shape:", model_data.shape)
display(model_data.head())

---
## 3. Exploratory Data Analysis (EDA)

EDA helps us understand cancellation frequency, feature distributions, and possible relationships between order characteristics and cancellation risk.

In [ ]:
# ── EDA 3.1 : Target variable distribution ─────────────────────────────────
plt.figure(figsize=(6, 4))
sns.countplot(data=model_data, x='IsCancelled', palette='BuPu')
plt.title('Order Cancellation Distribution')
plt.xlabel('Is Cancelled')
plt.ylabel('Order Count')
plt.show()

print("Cancellation rate:")
print((model_data['IsCancelled'].value_counts(normalize=True) * 100).round(2))

### Insight:
The target is imbalanced because most orders are not cancelled.  
Therefore, accuracy alone is not enough. We must also evaluate **Precision, Recall, F1-score, and ROC-AUC**.

In [ ]:
# ── EDA 3.2 : Distribution of numerical features ───────────────────────────
num_features_for_eda = [
    'TotalQuantity', 'TotalAmount', 'AvgUnitPrice', 'MaxUnitPrice',
    'UniqueProducts', 'TotalLines', 'CustomerOrderCount',
    'CustomerCancellationRate', 'CustomerAvgAmount'
]

model_data[num_features_for_eda].hist(bins=20, figsize=(18, 12), color='mediumpurple', edgecolor='white')
plt.suptitle('Distribution of Numerical Features', fontsize=16)
plt.show()

### Insight:
Several transaction-value variables are right-skewed, which is common in e-commerce because a small number of large orders can dominate total sales.

In [ ]:
# ── EDA 3.3 : Cancellation rate by country ─────────────────────────────────
country_cancel = model_data.groupby('Country')['IsCancelled'].agg(['mean', 'count']).reset_index()
country_cancel = country_cancel[country_cancel['count'] >= 20].sort_values('mean', ascending=False).head(15)

plt.figure(figsize=(12, 5))
sns.barplot(data=country_cancel, x='Country', y='mean', palette='Spectral')
plt.title('Top Countries by Cancellation Rate')
plt.ylabel('Cancellation Rate')
plt.xlabel('Country')
plt.xticks(rotation=45, ha='right')
plt.show()

display(country_cancel)

### Insight:
Cancellation risk may vary by country. Countries with very few orders should be interpreted carefully because small samples can exaggerate cancellation rates.

In [ ]:
# ── EDA 3.4 : Cancellation by day of week ──────────────────────────────────
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_cancel = model_data.groupby('InvoiceDayOfWeek')['IsCancelled'].mean().reindex(day_order)

plt.figure(figsize=(10, 4))
sns.barplot(x=day_cancel.index, y=day_cancel.values, color='crimson')
plt.title('Cancellation Rate by Day of Week')
plt.ylabel('Cancellation Rate')
plt.xlabel('Day of Week')
plt.xticks(rotation=30)
plt.show()

### Insight:
Order timing features can help detect operational or behavioral patterns linked to cancellation risk.

In [ ]:
# ── EDA 3.5 : Boxplots — numerical features by cancellation status ─────────
selected_box_cols = ['TotalAmount', 'TotalQuantity', 'UniqueProducts', 'TotalLines']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, col in zip(axes, selected_box_cols):
    sns.boxplot(data=model_data, x='IsCancelled', y=col, ax=ax, palette='Set2')
    ax.set_title(f'{col} by Cancellation Status')
    ax.set_xlabel('Is Cancelled')

plt.tight_layout()
plt.show()

### Insight:
Order size, product variety, and order amount may differ between cancelled and non-cancelled orders.

In [ ]:
# ── EDA 3.6 : Correlation heatmap ──────────────────────────────────────────
numeric_cols = model_data.select_dtypes(include=['int64', 'float64']).columns

plt.figure(figsize=(12, 8))
sns.heatmap(model_data[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap — Numerical Features')
plt.show()

### Insight:
Correlation helps identify linear relationships, but cancellation risk can also depend on nonlinear patterns. That is why tree-based models are included.

---
## 4. Feature Selection

Feature selection identifies which variables contribute most to predicting cancellation risk.

We use:
1. **Mutual Information** — captures nonlinear dependency between features and target.
2. **Random Forest Feature Importance** — captures tree-based predictive importance.

In [ ]:
# ── Train/test split before scaling, encoding, and model fitting ───────────
X_raw = model_data.drop('IsCancelled', axis=1)
y = model_data['IsCancelled']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=== Train / Test Split ===")
print(f"Total samples   : {len(X_raw)}")
print(f"Training samples: {len(X_train_raw)} ({len(X_train_raw)/len(X_raw)*100:.1f}%)")
print(f"Testing samples : {len(X_test_raw)} ({len(X_test_raw)/len(X_raw)*100:.1f}%)")
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

In [ ]:
# ── Identify numerical and categorical columns ─────────────────────────────
numeric_features = X_train_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_raw.select_dtypes(include=['object']).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

In [ ]:
# ── Preprocessing pipelines ────────────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [ ]:
# ── Feature Selection 4.1 : Mutual Information ─────────────────────────────
# Fit preprocessing only on training data to avoid test leakage.
X_train_processed = preprocessor.fit_transform(X_train_raw)

feature_names = preprocessor.get_feature_names_out()
X_train_dense = X_train_processed.toarray() if hasattr(X_train_processed, 'toarray') else X_train_processed

mi_scores = mutual_info_classif(X_train_dense, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
mi_series.head(15).sort_values().plot(kind='barh')
plt.title('Top 15 Features by Mutual Information')
plt.xlabel('Mutual Information Score')
plt.show()

display(mi_series.head(15).reset_index().rename(columns={'index': 'Feature', 0: 'MI_Score'}))

In [ ]:
# ── Feature Selection 4.2 : Random Forest Feature Importance ───────────────
rf_selector = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_selector.fit(X_train_processed, y_train)

rf_importance = pd.Series(
    rf_selector.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))

top_rf_importance = rf_importance.head(15).sort_values() # Get top 15 and sort ascending for barh plot

# Applying user's color logic: note that since feature importances are non-negative,
# the condition 'v > 0' will always be true, making all bars the first color.
colors = ['#e74c3c' if v > 0 else '#3498db' for v in top_rf_importance.values]

plt.barh(top_rf_importance.index, top_rf_importance.values, color=colors)
plt.title('Top 15 Features by Random Forest Importance')
plt.xlabel('Importance')
plt.show()

display(rf_importance.head(15).reset_index().rename(columns={'index': 'Feature', 0: 'Importance'}))

In [ ]:
# ── Feature Selection 4.3 : Final feature interpretation ───────────────────
common_features = list(set(mi_series.head(20).index) & set(rf_importance.head(20).index))

print(f"Common important features from both methods: {len(common_features)}")
for feature in common_features:
    print("-", feature)

Both Mutual Information and Random Forest importance are used to understand predictive drivers.  
For final modeling, the full processed feature set is kept because the dataset has a manageable number of variables after one-hot encoding.

---
## 5. Model Building

We train four classification models and evaluate them before optimization:

| # | Model | Why chosen |
|---|---|---|
| 1 | Logistic Regression | Simple, interpretable baseline |
| 2 | Decision Tree | Captures nonlinear rules |
| 3 | Random Forest | Strong ensemble model, handles nonlinear patterns |
| 4 | Support Vector Machine | Effective classifier with nonlinear boundary |

In [ ]:
# ── Helper function : compute classification metrics ──────────────────────
results_before = []

def evaluate_model(y_true, y_pred, y_proba, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_proba) if y_proba is not None else np.nan

    result = {
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'ROC_AUC': roc_auc
    }

    print(f"\n=== {model_name} ===")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

    return result

In [ ]:
# ── Model 5.1 : Logistic Regression ────────────────────────────────────────
logreg = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

logreg.fit(X_train_raw, y_train)
y_pred_lr = logreg.predict(X_test_raw)
y_proba_lr = logreg.predict_proba(X_test_raw)[:, 1]

res_lr = evaluate_model(y_test, y_pred_lr, y_proba_lr, 'Logistic Regression')
results_before.append(res_lr)

In [ ]:
# ── Model 5.2 : Decision Tree Classifier ───────────────────────────────────
dt = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42, class_weight='balanced'))
])

dt.fit(X_train_raw, y_train)
y_pred_dt = dt.predict(X_test_raw)
y_proba_dt = dt.predict_proba(X_test_raw)[:, 1]

res_dt = evaluate_model(y_test, y_pred_dt, y_proba_dt, 'Decision Tree Classifier')
results_before.append(res_dt)

In [ ]:
# ── Model 5.3 : Random Forest Classifier ───────────────────────────────────
rf = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    ))
])

rf.fit(X_train_raw, y_train)
y_pred_rf = rf.predict(X_test_raw)
y_proba_rf = rf.predict_proba(X_test_raw)[:, 1]

res_rf = evaluate_model(y_test, y_pred_rf, y_proba_rf, 'Random Forest Classifier')
results_before.append(res_rf)

In [ ]:
# ── Model 5.4 : Support Vector Machine (SVM) ───────────────────────────────
svm = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', SVC(
        kernel='rbf',
        probability=True,
        class_weight='balanced',
        random_state=42
    ))
])

svm.fit(X_train_raw, y_train)
y_pred_svm = svm.predict(X_test_raw)
y_proba_svm = svm.predict_proba(X_test_raw)[:, 1]

res_svm = evaluate_model(y_test, y_pred_svm, y_proba_svm, 'Support Vector Machine')
results_before.append(res_svm)

In [ ]:
# ── Model comparison before optimization ───────────────────────────────────
results_df = pd.DataFrame(results_before)
results_df = results_df.sort_values(by='ROC_AUC', ascending=False)

print("\n=== Model Comparison (Before Optimization) ===")
display(results_df)

best_model_before = results_df.iloc[0]
print(f"\nBest model before optimization: {best_model_before['Model']}")

---
## 6. Model Optimization

Optimization improves model performance by tuning hyperparameters.

We optimize:
- Decision Tree
- Random Forest

`scoring='f1'` is used because cancellation risk is an imbalanced classification problem and the minority class matters.

In [ ]:
# ── Optimization 6.1 : Decision Tree — GridSearchCV ────────────────────────
print("Running GridSearchCV for Decision Tree...")

dt_param_grid = {
    'model__max_depth': [3, 5, 10, 15, None],
    'model__min_samples_split': [2, 5, 10, 20],
    'model__min_samples_leaf': [1, 2, 5, 10],
    'model__criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(
    estimator=dt,
    param_grid=dt_param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

dt_grid.fit(X_train_raw, y_train)

print("Best Decision Tree parameters:")
print(dt_grid.best_params_)
print(f"Best CV F1-score: {dt_grid.best_score_:.4f}")

In [ ]:
# ── Optimization 6.2 : Decision Tree — evaluate optimized model ────────────
dt_best = dt_grid.best_estimator_

y_pred_dt_opt = dt_best.predict(X_test_raw)
y_proba_dt_opt = dt_best.predict_proba(X_test_raw)[:, 1]

res_dt_opt = evaluate_model(y_test, y_pred_dt_opt, y_proba_dt_opt, 'Decision Tree (Optimized)')

In [ ]:
# ── Optimization 6.3 : Random Forest — GridSearchCV ────────────────────────
print("Running GridSearchCV for Random Forest...")

rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, 20, None],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 5],
    'model__max_features': ['sqrt', 'log2']
}

rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train_raw, y_train)

print("Best Random Forest parameters:")
print(rf_grid.best_params_)
print(f"Best CV F1-score: {rf_grid.best_score_:.4f}")

In [ ]:
# ── Optimization 6.4 : Random Forest — evaluate optimized model ────────────
rf_best = rf_grid.best_estimator_

y_pred_rf_opt = rf_best.predict(X_test_raw)
y_proba_rf_opt = rf_best.predict_proba(X_test_raw)[:, 1]

res_rf_opt = evaluate_model(y_test, y_pred_rf_opt, y_proba_rf_opt, 'Random Forest (Optimized)')

In [ ]:
# ── Optimization 6.5 : Cross-validation scores ────────────────────────────
print("=== 5-Fold Cross-Validation F1 Scores ===\n")

models_for_cv = [
    (logreg, 'Logistic Regression'),
    (dt_best, 'Decision Tree Optimized'),
    (rf_best, 'Random Forest Optimized'),
    (svm, 'Support Vector Machine')
]

for model, name in models_for_cv:
    scores = cross_val_score(model, X_train_raw, y_train, cv=5, scoring='f1', n_jobs=-1)
    print(f"{name}: Mean F1 = {scores.mean():.4f}, Std = {scores.std():.4f}")

---
## 7. Model Evaluation & Validation

We compare all models using classification metrics:

| Metric | Meaning | Best Direction |
|---|---|---|
| Accuracy | Overall correct predictions | Higher |
| Precision | Of predicted cancellations, how many were actual cancellations | Higher |
| Recall | Of actual cancellations, how many were detected | Higher |
| F1-score | Balance between precision and recall | Higher |
| ROC-AUC | Ranking ability across thresholds | Higher |

In [ ]:
# ── Evaluation 7.1 : Full comparison table ─────────────────────────────────
all_results = [
    res_lr,
    res_dt, res_dt_opt,
    res_rf, res_rf_opt,
    res_svm
]

results_df = pd.DataFrame(all_results).set_index('Model')
results_df = results_df.sort_values(by='F1', ascending=False)

print("=== Final Model Comparison ===")
display(results_df)

In [ ]:
# ── Evaluation 7.2 : Visual comparison ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

plot_df = results_df.reset_index()
short_names = plot_df['Model'].str.replace(' Classifier', '').str.replace('Support Vector Machine', 'SVM')

sns.barplot(data=plot_df, x=short_names, y='Precision', ax=axes[0])
axes[0].set_title('Precision Comparison')
axes[0].tick_params(axis='x', rotation=80)

sns.barplot(data=plot_df, x=short_names, y='Recall', ax=axes[1])
axes[1].set_title('Recall Comparison')
axes[1].tick_params(axis='x', rotation=80)

sns.barplot(data=plot_df, x=short_names, y='F1', ax=axes[2])
axes[2].set_title('F1-score Comparison')
axes[2].tick_params(axis='x', rotation=80)

plt.tight_layout()
plt.show()

In [ ]:
# ── Evaluation 7.3 : Best model — confusion matrix and classification report
best_model_name = results_df['F1'].idxmax()

model_lookup = {
    'Logistic Regression': (logreg, y_pred_lr, y_proba_lr),
    'Decision Tree Classifier': (dt, y_pred_dt, y_proba_dt),
    'Decision Tree (Optimized)': (dt_best, y_pred_dt_opt, y_proba_dt_opt),
    'Random Forest Classifier': (rf, y_pred_rf, y_proba_rf),
    'Random Forest (Optimized)': (rf_best, y_pred_rf_opt, y_proba_rf_opt),
    'Support Vector Machine': (svm, y_pred_svm, y_proba_svm)
}

best_model, best_pred, best_proba = model_lookup[best_model_name]

print(f"Best model selected by F1-score: {best_model_name}")
print("\nClassification Report:")
print(classification_report(y_test, best_pred, target_names=['Not Cancelled', 'Cancelled']))

cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Cancelled', 'Cancelled'],
            yticklabels=['Not Cancelled', 'Cancelled'])
plt.title(f'Confusion Matrix — {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# ── Evaluation 7.4 : ROC Curve ─────────────────────────────────────────────
plt.figure(figsize=(8,6))

for model_name, (_, _, y_proba) in model_lookup.items():
    if y_proba is not None:
        RocCurveDisplay.from_predictions(y_test, y_proba, name=model_name, ax=plt.gca())

plt.title('ROC Curve Comparison')
plt.show()

### Evaluation Insight:
For cancellation-risk prediction, **Recall** is especially important when the business wants to catch as many risky orders as possible.  
However, very high recall with low precision can waste operational resources. The final model should be selected based on the business trade-off between missed cancellations and false alarms.

---
## 8. Business Insights & Recommendations

The model can support proactive cancellation-risk management, customer segmentation, and operational planning.

In [ ]:
# ── Insights 8.1 : Top risk drivers from optimized Random Forest ───────────
# Use optimized Random Forest for feature importance interpretation.

rf_final = rf_best if 'rf_best' in globals() else rf
rf_model = rf_final.named_steps['model']
rf_preprocessor = rf_final.named_steps['preprocess']

feature_names_final = rf_preprocessor.get_feature_names_out()
importances_final = pd.Series(
    rf_model.feature_importances_,
    index=feature_names_final
).sort_values(ascending=False)

top_importances = importances_final.head(15)

plt.figure(figsize=(10, 6))
top_importances.sort_values().plot(kind='barh')
plt.title('Top 15 Cancellation Risk Drivers — Random Forest')
plt.xlabel('Feature Importance')
plt.show()

display(top_importances.reset_index().rename(columns={'index': 'Feature', 0: 'Importance'}))

### Insight:
The most important features identify which order characteristics are associated with cancellation risk.  
Business teams can use these signals to prioritize review, customer communication, and inventory checks.

In [ ]:
# ── Insights 8.2 : Cancellation risk bands ─────────────────────────────────
risk_df = X_test_raw.copy()
risk_df['ActualCancellation'] = y_test.values
risk_df['PredictedRiskProbability'] = best_proba

risk_df['RiskBand'] = pd.cut(
    risk_df['PredictedRiskProbability'],
    bins=[0, 0.25, 0.50, 0.75, 1.00],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical Risk'],
    include_lowest=True
)

risk_summary = risk_df.groupby('RiskBand').agg(
    Orders=('ActualCancellation', 'count'),
    ActualCancellationRate=('ActualCancellation', 'mean'),
    AvgPredictedRisk=('PredictedRiskProbability', 'mean')
).reset_index()

display(risk_summary)

plt.figure(figsize=(8, 5))
sns.barplot(data=risk_summary, x='RiskBand', y='ActualCancellationRate')
plt.title('Actual Cancellation Rate by Predicted Risk Band')
plt.ylabel('Actual Cancellation Rate')
plt.xlabel('Predicted Risk Band')
plt.show()

### Business Use:
Risk bands make the model actionable:
- **Low Risk:** normal processing
- **Medium Risk:** light monitoring
- **High Risk:** proactive confirmation or customer follow-up
- **Critical Risk:** urgent review before fulfillment

In [ ]:
# ── Insights 8.3 : Country-level operational recommendation ───────────────
country_risk = risk_df.groupby('Country').agg(
    Orders=('ActualCancellation', 'count'),
    ActualCancellationRate=('ActualCancellation', 'mean'),
    AvgPredictedRisk=('PredictedRiskProbability', 'mean')
).reset_index()

country_risk = country_risk[country_risk['Orders'] >= 20]
country_risk = country_risk.sort_values('AvgPredictedRisk', ascending=False).head(10)

display(country_risk)

plt.figure(figsize=(12, 5))
sns.barplot(data=country_risk, x='Country', y='AvgPredictedRisk')
plt.title('Top Countries by Average Predicted Cancellation Risk')
plt.ylabel('Average Predicted Risk')
plt.xlabel('Country')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────
best_model_row = results_df.loc[results_df['F1'].idxmax()]

print("Final model selected based on highest F1-score\n")
print("╔══════════════════════════════════════════════╗")
print("║       ORDER CANCELLATION RISK MODEL         ║")
print("╚══════════════════════════════════════════════╝")
print(f"Best Model : {best_model_row.name}")
print(f"Accuracy   : {best_model_row['Accuracy']:.4f}")
print(f"Precision  : {best_model_row['Precision']:.4f}")
print(f"Recall     : {best_model_row['Recall']:.4f}")
print(f"F1-score   : {best_model_row['F1']:.4f}")
print(f"ROC-AUC    : {best_model_row['ROC_AUC']:.4f}")

### Final Conclusion

This project successfully applied machine learning to predict **order cancellation risk** in e-commerce.

The workflow included:
- Converting transaction-line data into invoice/order-level data
- Creating the cancellation target from invoice numbers
- Performing EDA to understand cancellation patterns
- Building multiple classification models
- Optimizing tree-based models
- Evaluating models with business-relevant classification metrics
- Translating predictions into practical risk bands

The final model can help an e-commerce business proactively identify risky orders and reduce cancellation-related operational loss.